In [ ]:
import torch
from diffusers import AudioLDM2Pipeline
from transformers import GPT2LMHeadModel
import scipy.io.wavfile
import numpy as np # <-- Importamos numpy para formatear el audio

# 1. CONFIGURACIÓN
device = "cuda" if torch.cuda.is_available() else "cpu"
# CAMBIO CLAVE 1: Forzamos float32 siempre para evitar audios "vacíos"
dtype = torch.float32
print(f"🎸 Entorno de Audio listo: {device} (Precisión: {dtype})")

repo_id = "cvssp/audioldm2-music"

# 2. CARGA BASE
print("Cargando pipeline base...")
pipe = AudioLDM2Pipeline.from_pretrained(repo_id, torch_dtype=dtype)

# 3. EL PARCHE
print("Inyectando el modelo de lenguaje correcto en memoria...")
pipe.language_model = GPT2LMHeadModel.from_pretrained(
    repo_id,
    subfolder="language_model",
    torch_dtype=dtype
)
pipe.to(device)

# 4. GENERACIÓN
prompt = "Traditional Colombian cumbia scene, vibrant Caribbean coastal setting, dancers in flowing white and red dresses, men in sombrero vueltiao hats, barefoot on a wooden stage, accordion, guacharaca, tambora and maracas, warm sunset lighting, festive and joyful atmosphere, rhythmic movement, folkloric authenticity, cinematic detail"
negative_promt = "No modern electronic instruments, no urban or reggaeton elements, no rock guitars, no futuristic clothing, no neon lights, no non-Colombian cultural references, no distortion of traditional costumes, no dull or muted colors"
print("Sintetizando audio... (esto toma unos segundos)")
audio = pipe(
    prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=50,
    audio_length_in_s=10.0
).audios[0]

# 5. DIAGNÓSTICO Y GUARDADO
path_salida = "musica_generada.wav"

# Revisamos qué produjo exactamente la gráfica
volumen_maximo = np.max(np.abs(audio))

if np.isnan(volumen_maximo) or volumen_maximo == 0.0:
    print("❌ ERROR: La tarjeta gráfica generó un archivo matemáticamente vacío.")
else:
    print(f"✅ ¡Sonido detectado! (Nivel máximo: {volumen_maximo})")

    # CAMBIO CLAVE 2: Convertimos al formato de audio estándar que leen todos los reproductores
    audio_normalizado = audio / volumen_maximo
    audio_estandar = (audio_normalizado * 32767).astype(np.int16)

    scipy.io.wavfile.write(path_salida, rate=16000, data=audio_estandar)
    print(f"✨ ¡Listo! Audio formateado correctamente. Escucha tu creación en: {path_salida}")